In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

transactions_clean = pd.read_csv("clean_data/transactions_clean.csv", parse_dates=["invoice_date"])
customers_clean    = pd.read_csv("clean_data/customers_clean.csv")

print("Données chargées ✓")

Données chargées ✓


In [30]:
print("\n\n------------------------------------------------ ETAPE 1 : Construction des features RFM ----------------------------------------------")
print("\n1) À partir de transactions.csv nettoyé, calculez pour chaque client Recence / fréquence / montant")

# La snapshot date = date max du dataset (pas la date du jour)
snapshot_date = transactions_clean["invoice_date"].max()
print(f"\nSnapshot date : {snapshot_date.date()}")

print("\n====== CALCUL RFM ======")
rfm = transactions_clean.groupby("customer_id").agg(
    recence   = ("invoice_date", lambda dates: (snapshot_date - dates.max()).days),
    frequence = ("invoice_id",   "nunique"),
    montant   = ("line_total",   "sum")
).reset_index()
print(f"\nNombre de clients dans la table RFM : {len(rfm)}")
print("\nAperçu :")
print(rfm.head(20).to_string(index=False))

print("\n====== TOP 15 PAR RÉCENCE (plus récent → moins récent) ======")
print(rfm.sort_values("recence",
ascending=True).head(15).to_string(index=False))

print("\n====== TOP 15 PAR FRÉQUENCE (plus fréquent → moins fréquent) ======")
print(rfm.sort_values("frequence",
ascending=False).head(15).to_string(index=False))

print("\n====== TOP 15 PAR MONTANT (plus gros → plus petit) ======")
print(rfm.sort_values("montant",
ascending=False).head(15).to_string(index=False))

print("\n====== FEATURES SUPPLEMENTAIRES ======")
print("--> Ajout du panier moyen et de l'ancienneté (tenure)")
rfm["panier_moyen"] = rfm["montant"] / rfm["frequence"]

customers_clean["first_purchase"] = pd.to_datetime(customers_clean["first_purchase"])
anciennete = customers_clean[["customer_id", "first_purchase"]].copy()
anciennete["anciennete_jours"] = (snapshot_date - anciennete["first_purchase"]).dt.days
rfm = rfm.merge(anciennete[["customer_id", "anciennete_jours"]], on="customer_id", how="left")

print("\nStatistiques :")
print(rfm[["recence", "frequence", "montant", "panier_moyen", "anciennete_jours"]].describe().round(1))



------------------------------------------------ ETAPE 1 : Construction des features RFM ----------------------------------------------

1) À partir de transactions.csv nettoyé, calculez pour chaque client Recence / fréquence / montant

Snapshot date : 2026-06-30

====== CALCUL RFM ======

Nombre de clients dans la table RFM : 47662

Aperçu :
 customer_id  recence  frequence  montant
     12346.0      325          3   500.56
     12347.0        1          8  6472.20
     12348.0       74          5  1537.84
     12349.0       18          3  5085.38
     12350.0      309          1   459.45
     12351.0      374          1   632.28
     12352.0       35          9  3096.61
     12353.0      203          2   780.89
     12354.0      231          1  2083.84
     12355.0      213          2   900.58
     12356.0       22          6  3517.13
     12358.0        1          5  1741.74
     12360.0       51          6  6032.58
     12361.0      286          4  1179.56
     12362.0        2  